# D5 — Win Rate by Z-Score Bucket

In [ ]:
import sys, gc
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
sys.path.insert(0, '.')
sys.path.insert(0, '../notebooks')
from config import (RESULTS_DIR, SEAGATE_DIR, FIGS_DIR, WINDOWS_META,
                    BASELINE_STATS, UPDATED_STATS, WIN_COLORS,
                    TICK, MULT, save_fig)
Path('figures').mkdir(exist_ok=True)


In [ ]:
bins = [0, 1, 2, 3, 4, np.inf]
labels = ['0-1','1-2','2-3','3-4','4+']

fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharey=True)
for ax, direction in zip(axes, ['Long', 'Short']):
    for wk, wm in WINDOWS_META.items():
        rk = wm['result_key']
        trades = pd.read_parquet(RESULTS_DIR / rk / 'none' / 'trades.parquet')
        sub = trades[trades['dir_label'] == direction].copy()
        sub['z_bucket'] = pd.cut(sub['entry_z'].abs(), bins=bins, labels=labels, right=False)
        wr = sub.groupby('z_bucket', observed=True).apply(lambda g: (g['gross_usd']>0).mean())
        ax.plot(range(len(labels)), wr.reindex(labels).values,
                marker='o', label=wk, color=WIN_COLORS[wk])
    ax.axhline(0.5, color='black', ls='--', lw=0.8)
    ax.set_xticks(range(len(labels))); ax.set_xticklabels(labels)
    ax.set_xlabel('|Entry Z| Bucket')
    ax.set_title(f'Win Rate by Z-Score Bucket — {direction}')
    ax.legend(fontsize=8)
axes[0].set_ylabel('Win Rate')
fig.suptitle('Win Rate by Entry Z-Score Magnitude', fontsize=13)
save_fig(fig, 'D5_wr_by_zscore_bucket.png')
